In [ ]:

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.ticker as mticker


In [ ]:
path = '/Volumes/Extreme SSD/PhD backup/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/PIC_discrete/all_PIC.csv'
ds_pic = pd.read_csv(path)

print(ds_pic.columns)

In [ ]:

# ds_pic columns: lat, lon, datetime, pic
ds_pic = ds_pic.copy()


fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.65, 0.35],
    specs=[[{"type": "scattermap"}, {"type": "xy"}]],
    subplot_titles=("PIC locations", "PIC distribution")
)

fig.add_trace(
    go.Scattermap(
        lat=ds_pic["lat"],
        lon=ds_pic["lon"],
        mode="markers",
        marker=dict(
            size=7,
            color=ds_pic["PIC_mmolC_m3"],
            colorscale="Viridis",
            showscale=True,
            colorbar=dict(title="PIC"),
            cmin = 0.00,
            cmax = 4

        ),
        customdata=ds_pic[["datetime", "PIC_mmolC_m3", "lat", "lon"]],
        hovertemplate=(
            "Datetime: %{customdata[0]|%Y-%m-%d %H:%M:%S}<br>"
            "PIC: %{customdata[1]:.3g}<br>"
            "Lat: %{customdata[2]:.3f}<br>"
            "Lon: %{customdata[3]:.3f}<br>"
            "<extra></extra>"
        ),
        name="Observations"
    ),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(
        x=ds_pic["PIC_mmolC_m3"],
        nbinsx=50,
        marker=dict(color="steelblue"),
        name="PIC histogram"
    ),
    row=1, col=2
)

fig.update_layout(
    height=600,
    map=dict(
        style="carto-positron",
        zoom=1
    ),
    margin=dict(l=20, r=20, t=50, b=20),
    hovermode="closest"
)

fig.update_xaxes(title_text="PIC", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=2)

fig.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature


import numpy as np
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature


def plot_locations_lambert(
    ds,
    lat="lat",
    lon="lon",
    figsize=(10, 7),
    marker_color="tab:blue",
    marker_size=35,
    marker_alpha=0.8,
    title="PIC sampling locations",
    extent=None,
    standard_parallels=None,
    savedir = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Figures'
):
    """
    Plot latitude/longitude locations using a Lambert Conformal projection.
    """

    if lat not in ds.columns or lon not in ds.columns:
        raise ValueError(
            f"Dataset must contain '{lat}' and '{lon}' columns."
        )

    data = ds[[lat, lon]].copy()
    data = data.replace([np.inf, -np.inf], np.nan).dropna()

    if data.empty:
        raise ValueError("No valid latitude/longitude values found.")

    lat_values = data[lat].to_numpy()
    lon_values = data[lon].to_numpy()

    central_longitude = float(np.mean(lon_values))
    central_latitude = float(np.mean(lat_values))

    if standard_parallels is None:
        lat_min = np.nanmin(lat_values)
        lat_max = np.nanmax(lat_values)

        # Put standard parallels near the lower and upper parts
        # of the data range.
        standard_parallels = (
            lat_min + 0.25 * (lat_max - lat_min),
            lat_min + 0.75 * (lat_max - lat_min),
        )

    projection = ccrs.LambertConformal(
        central_longitude=central_longitude,
        central_latitude=central_latitude,
        standard_parallels=standard_parallels,
    )

    fig = plt.figure(figsize=figsize)

    ax = fig.add_subplot(
        1,
        1,
        1,
        projection=projection,
    )

    ax.add_feature(
        cfeature.OCEAN,
        facecolor="white",
        zorder=0,
    )

    ax.add_feature(
        cfeature.LAND,
        facecolor="0.92",
        edgecolor="0.75",
        linewidth=0.5,
        zorder=0,
    )

    ax.coastlines(
        resolution="10m",
        linewidth=0.4,
        color="0.25",
    )

    ax.scatter(
        lon_values,
        lat_values,
        s=marker_size,
        color=marker_color,
        alpha=marker_alpha,
        edgecolor="white",
        linewidth=0.5,
        transform=ccrs.PlateCarree(),
        zorder=3,
    )

    if extent is None:
        lon_range = lon_values.max() - lon_values.min()
        lat_range = lat_values.max() - lat_values.min()

        lon_margin = max(lon_range * 0.05, 2)
        lat_margin = max(lat_range * 0.05, 2)

        extent = [
            lon_values.min() - lon_margin,
            lon_values.max() + lon_margin,
            lat_values.min() - lat_margin,
            lat_values.max() + lat_margin,
        ]

    ax.set_extent(
        extent,
        crs=ccrs.PlateCarree(),
    )

    gridlines = ax.gridlines(
    crs=ccrs.PlateCarree(),
    draw_labels=True,
    linewidth=0.4,
    color="0.5",
    alpha=0.5,
    linestyle="--",
    x_inline=False,
    y_inline=False,
)

    gridlines.xlocator = mticker.FixedLocator(
        np.arange(-180, 181, 20)
    )

    gridlines.ylocator = mticker.FixedLocator(
        np.arange(0, 91, 10)
    )

    gridlines.top_labels = False
    gridlines.right_labels = False

    ax.set_title(title, pad=10)

    filename = f'{savedir}_{title}.png'

    plt.savefig(filename, bbox_inches="tight",dpi = 600)

    return fig, ax

ds_pic = ds_pic.copy()

plot_locations_lambert(
    ds_pic,
    lat="lat",
    lon="lon",
    figsize=(10, 7),
    marker_color="tab:red",
    marker_size=15,
    marker_alpha=0.8,
    title="PIC sampling locations",
    extent=[-70,10,40,80],
    standard_parallels=(45, 70),
)

In [ ]:
import matplotlib.pyplot as plt

def plot_locations_plain(ds, lat="lat", lon="lon"):
    fig, ax = plt.subplots(figsize=(9, 6))

    ax.scatter(
        ds[lon],
        ds[lat],
        color="tab:blue",
        s=35,
        alpha=0.8,
        edgecolor="white",
        linewidth=0.5,
    )

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("PIC sampling locations")
    ax.grid(alpha=0.3)

    return fig, ax